Read downloaded Hazel results

Set `RUNS` to the extracted result folder. This notebook reads completed outputs and never launches training. Partial pilot results are explicitly identified. No figures/tables are saved until the final export cell is enabled.

In [ ]:
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
from hazel_gp.results import (load_results, lolo_report, plot_parity,
                              plot_lolo_comparison, grouped_predictions)

ROOT = Path.cwd().resolve()
RUNS = ROOT / "data_hazel/returned/experiment_v1"
# For a pilot, point RUNS to its extracted folder instead.
tables, collection = load_results(RUNS)
print("Complete benchmark:", collection["complete"])
print("Completed tasks:", collection["completed_tasks"], "/", collection["expected_tasks"])
display(tables["summary"])

### Inspect predictions
Use the dropdowns to select model, evaluation method, and reference ligand. For matched IID, a reference ligand labels the size-matching comparison; its test set contains multiple ligands.

In [ ]:
from hazel_gp.notebook import review_controls
display(review_controls(tables))

### LOLO reported three ways
`lolo_report` returns each held-out ligand on its own, the unweighted mean across the eight ligand-left-out models, and the pooled score after all eight folds. Pooled and averaged values are not interchangeable: pooled R² is measured against the variance of the whole dataset, a per-ligand R² against the variance within that one ligand, so the averaged value is systematically harsher.

Matched IID now uses the same eight-fold geometry with random membership, so its pooled score is the like-for-like in-distribution comparison.

In [ ]:
METRIC = "rmse"
report = lolo_report(tables["predictions"], tables["metrics_by_split"])
display(report[["model", "view", "scope", "n_test", "r2", "rmse", "mae", "kendall_tau"]].round(3))

fig_ligands = plot_lolo_comparison(tables["metrics_by_split"], metric=METRIC)
display(fig_ligands)

### Grouped arrays, if needed
Seeds are recorded in the split metadata; arrays are stored as lists per reference group.

In [ ]:
grouped_iid = grouped_predictions(tables["predictions"], model="selected_2", method="iid_matched")
print("Reference groups:", list(grouped_iid))
# Example after the IID tasks have completed:
# grouped_iid["SPhos"]["y_pred"]

### Final export
Set `SAVE = True`, choose a **new** output folder, select the formats/tables, and execute this cell. Existing destinations are protected.

In [ ]:
SAVE = False
EXPORT_DIR = ROOT / "exports/local_review_v1"
FIGURE_FORMATS = ["png", "pdf"]
TABLES_TO_SAVE = ["summary", "metrics_by_split", "predictions"]

if SAVE:
    EXPORT_DIR.mkdir(parents=True, exist_ok=False)
    for name in TABLES_TO_SAVE:
        tables[name].to_csv(EXPORT_DIR / f"{name}.csv", index=False)
    report.to_csv(EXPORT_DIR / "lolo_report.csv", index=False)
    for name, fig in {"lolo_by_ligand": fig_ligands}.items():
        for fmt in FIGURE_FORMATS:
            fig.savefig(EXPORT_DIR / f"{name}.{fmt}", dpi=300, bbox_inches="tight")
    print("Saved:", EXPORT_DIR)
else:
    print("Preview only. Set SAVE=True when ready.")